### hello 도구 테스트

In [2]:
# https://github.com/langchain-ai/langchain-mcp-adapters?tab=readme-ov-file#streamable-http

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

from langchain_mcp_adapters.tools import load_mcp_tools

async with streamablehttp_client("http://127.0.0.1:8000/mcp/") as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        # 'hello' 도구를 비동기적으로 실행하고 결과를 result 변수에 할당
        # MCP는 서버이기 때문에 네트워크로 비동기 통신함
        result = await tools[1].ainvoke({"name": "김이남"})
        print(result)

[{'type': 'text', 'text': '안녕하세요, 김이남님', 'id': 'lc_3945207e-83d1-40c1-8fb6-74246355f9a3'}]


In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "mcp_tools": {
            "url": "http://localhost:8000/mcp/",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()

In [4]:
tools

[StructuredTool(name='add', description='Add two numbers', args_schema={'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'_fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000024441A3BF60>),
 StructuredTool(name='hello', description='간단한 인사말을 반환하는 도구', args_schema={'properties': {'name': {'default': '아무개', 'type': 'string'}}, 'type': 'object'}, metadata={'_meta': {'_fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000024441A5BD80>),
 StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수\n\nArgs:\n    timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n    location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema={'properties': {'timezone': {'default': 'Asia/Seoul', 'type': 'stri

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [6]:
from typing import Annotated # annotated는 타입 힌트를 사용할 때 사용하는 함수
from typing_extensions import TypedDict # TypedDict는 딕셔너리 타입을 정의할 때 사용하는 함수

from langgraph.graph.message import add_messages

# 상태 정의
class State(TypedDict):	
    messages: Annotated[list[str], add_messages]

In [7]:
# * 도구 바인딩
def call_model(state: State):
    response = model.bind_tools(tools).invoke(state["messages"])
    return {"messages": response}

In [8]:
from langgraph.graph import START, END

def route_tools(state: State):
    """
    마지막 메시지에 도구 호출이 있는 경우 ToolNode로 라우팅하고,
    그렇지 않은 경우 END로 라우팅하기 위해 conditional_edge에서 사용합니다.
    """
    if isinstance(state, list):
        ai_message = state[-1]
    elif messages := state.get("messages", []):
        ai_message = messages[-1]
    else:
        raise ValueError(f"tool_edge 입력 상태에서 메시지를 찾을 수 없습니다: {state}")
        
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    return END

In [9]:
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolNode

graph_builder = StateGraph(State)

graph_builder.add_node("call_model", call_model)
graph_builder.add_node(ToolNode(tools))
graph_builder.add_edge(START, "call_model")
graph_builder.add_conditional_edges(
    "call_model",
    route_tools,
    {"tools": "tools", END: END}
)
graph_builder.add_edge("tools", "call_model")
graph = graph_builder.compile()

In [10]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [12]:
await graph.ainvoke({"messages": "지금 부산 몇시야?"})

{'messages': [HumanMessage(content='지금 부산 몇시야?', additional_kwargs={}, response_metadata={}, id='1726c0c7-a208-4dc8-b488-2343396c86f1'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "Busan"}'}, '__gemini_function_call_thought_signatures__': {'614e21d6-e33f-42b9-81a5-31bd5488c844': 'CrYCAXLI2nxPn6bF+gLBfEOyZ3lSdtm5dgz0cE41JiORj2OjToQyH+jTl07S87Hg9Fu5DfHadGTqGeRvngBMW2DGKkwjMtlh4JR9mESU0RgH42iTNWlUEesFSMCCyZT419hiCxA3hxhR2BswQEvKYyQVFkoOSLuBdacIVYDSJwNxku9b4FBqTXTbMAYOWxUFBlxdMTFfimD+2LVjqKjuMONaum5OGyVnyegbQCR0sJ2AiyOLktwYvRCoW2DWAzS0Xi1yFSN9tpuKGAyr61AyEPnJ4XiHjSwQIuWKD6ca7n+nKeyK9JdMKVDInr5xF2FwYjfUf/BV617603GX97kXjUo60HWvbl1QT9L297f//AmNtDycK+k4Cl948sRiP9j2DBf/eXXvInRxzgyAqPh43KD6AFZeeeRO0w=='}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}

In [13]:
stock_response = await graph.ainvoke({"messages": "테슬라 주가는 한달 전 보다 올랐나 내렸나?"})
stock_response

{'messages': [HumanMessage(content='테슬라 주가는 한달 전 보다 올랐나 내렸나?', additional_kwargs={}, response_metadata={}, id='07cfd66c-f48b-4faa-89f0-06c792aa956c'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"ticker": "TSLA", "period": "1mo"}}'}, '__gemini_function_call_thought_signatures__': {'e04ecb9b-6168-4a6e-8536-27eb7ab0d982': 'CsEVAXLI2nzVSABkofq1W9rWpI3HlAzbh9dRHZtIIjg6r2LMJ3EwArRjo6oYdlNzTdA08bxR4oGX2n1BYNjpVLfXIQEcpDDX4RML+HP9f5BwPwwpRlyHdecwvU7Jw4KGLBhu8cFn6nrOM8CDvbg3QuRP4R2josHAp2rzvva7sV8kLRlM2EN7sC+smKUIymNbykocmjIU7ogO524AWibG0wlDV/Q2HulPMKYn4JmMuFaLDE1Y30392M84iFJj4qA/+d6IkhjZRXzsealrfuWqevIR9+YmbCNR1sy3w4dSbS5DdDhR86PPqmiN24r/Col4ntqDgFDNK1RZzpf7wNhCkNWm4ZnFwtPrfvTNvvP2i8nhAH5XTS5i1v9E+U5wyvv97RKZ3HQ4K8jXF1odaebEfpUU5K8jSklxMGZ9B8TyKw3CsC+RHhQEQfphBvipbbQqQMyRfAs1uekkpN6G4CV6aii/E5Baw2I54ZNCcA9uY9uCvhXE7N0y0d/roiY3KaqzPbTGe41ZcmZjwqNbW++yuGe4YyJn7U5aOXzfLqlfdatNFHuEaZMToKCVdJ1H0UfnwOmOsPqloPYSZ